<div align="center">

# Week 07: Embedded Computing and Real-Time Scheduling: RM, EDF and the PLC Scan Cycle

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-07) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/Week07_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines `lab_rng`, the random generator of the interactive lab, and the delays of the programmable logic controller (PLC) of the main switchboard. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

def lab_rng(seed):
    """The random generator of the interactive lab, so that the Colab notebook draws exactly the same moments."""
    s = seed & 0xFFFFFFFF
    def uniform():
        nonlocal s
        s = (s + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((s ^ (s >> 15)) * (s | 1)) & 0xFFFFFFFF
        t = (t ^ ((t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF)) & 0xFFFFFFFF)) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    def normal():
        u = v = 0.0
        while u == 0.0:
            u = uniform()
        while v == 0.0:
            v = uniform()
        return math.sqrt(-2 * math.log(u)) * math.cos(2 * math.pi * v)
    return uniform, normal

# The PLC of the main switchboard: the input filter, the output relay, the required reaction and the watchdog, in ms
T_IN, T_OUT, LIMIT_MS, WATCHDOG_MS = 3.0, 6.0, 50.0, 30.0
COLORS = ["#B53F2E", "#1E6FA8", "#2E8B57", "#C98510", "#7B5EA7", "#56616D"]
print("Setup complete. Running in Colab:", IN_COLAB, "| PLC: limit", LIMIT_MS, "ms, watchdog", WATCHDOG_MS, "ms")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 7.1": (lambda: 4 / 100, 1e-9),
    "Exercise 7.2": (lambda: 50, 1e-9),
    "Exercise 7.3": (lambda: 2, 1e-9),
    "Exercise A": (lambda: utilization(TASKS) + 4 / 100, 1e-9),
    "Exercise B": (lambda: response_times([Task('protection', 1, 5), Task('frequency control', 3, 10), Task('battery management', 6, 20), Task('Modbus', 9, 50)])['battery management']['R'], 1e-9),
    "Exercise C": (lambda: int(report(task_set(8), simulate(task_set(8), 'EDF')[1])['missed'][0]), 1e-9),
    "Exercise D": (lambda: max(k for k in range(1, 40) if worst_case(k) <= LIMIT_MS), 1e-9),
    "Exercise E": (lambda: utilization(task_set(7)), 1e-9),
}

## Python step 7: Tasks and time

The scheduling of this week works with tasks that have a name, an execution time and a period. This step writes a task as a class, sorts tasks by their period and runs a short scheduler loop, one millisecond at a time &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#ref-3)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "WATCHDOG_MS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-07/"
               "NB07_realtime_scheduling.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Classes: Objects of your own kind

In [ ]:
# Python practice: a small example for learning the language
class Task:
    """A periodic task: a name, an execution time C and a period T, both in ms."""
    def __init__(self, name, C, T):
        self.name = name
        self.C = C
        self.T = T

    def utilization(self):
        return self.C / self.T

bm = Task("battery management", 5, 20)          # one object of the class Task
print(bm.name, bm.C, bm.T, bm.utilization())

A class is a template for objects of one kind. `__init__` runs when an object is made and stores its values as attributes of `self`, the object itself. The method `utilization` is a function that belongs to the class and reads the attributes of its object.

### Sorting by priority

In [ ]:
# Python practice: a small example for learning the language
tasks = [Task("Modbus", 9, 50), Task("protection", 1, 5), Task("battery management", 5, 20),
         Task("frequency control", 3, 10)]
by_period = sorted(tasks, key=lambda t: t.T)     # the shortest period first
print([t.name for t in by_period])
print("total utilization:", round(sum(t.utilization() for t in tasks), 2))

`sorted` returns a new list in order. The argument `key` names what to compare, here with a `lambda`, a function of one line: For each task `t` it returns `t.T`. A rate-monotonic scheduler orders its tasks in exactly this way.

### A scheduler loop

In [ ]:
# Python practice: a small example for learning the language
left = {t.name: t.C for t in by_period}           # the work left of each task, in ms
timeline = []
for ms in range(20):
    running = next((t.name for t in by_period if left[t.name] > 0), "idle")
    if running != "idle":
        left[running] -= 1                        # the task runs for one millisecond
    timeline.append(running[:4])
print(" ".join(timeline))

Each pass of the loop is one millisecond. `next` returns the first task in the order of priority that still has work, or "idle". This loop releases each task only once, at 0 ms. The scheduler of Section 2 also releases every task again at each of its periods.

**Quick check.** In which millisecond does the battery management task start in the timeline above, and why not earlier?

<details><summary>Answer</summary>

In millisecond 4, counting from 0: Protection and frequency control have shorter periods, so they run first, for 1 and 3 ms.

</details>

### Exercises for Python step 7

**Exercise 7.1.** Make a `Task` for a logging task with `C = 4` ms and `T = 100` ms and store its utilization in `u_71`.

In [ ]:
u_71 = None   # your answer

check("Exercise 7.1", u_71)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
u_71 = Task("logging", 4, 100).utilization()

check("Exercise 7.1", u_71)

**Exercise 7.2.** Sort `tasks` by their execution time with the longest first and store the period of the first task in `t_72`.

In [ ]:
t_72 = None   # your answer

check("Exercise 7.2", t_72)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_72 = sorted(tasks, key=lambda t: t.C, reverse=True)[0].T

check("Exercise 7.2", t_72)

**Exercise 7.3.** Count the milliseconds of `timeline` in which the processor is idle and store the count in `i_73`.

In [ ]:
i_73 = None   # your answer

check("Exercise 7.3", i_73)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
i_73 = timeline.count("idle")

check("Exercise 7.3", i_73)

## 1. Tasks on one processor &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#ref-1)&#93;

*Lecture: [Tasks on one processor](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#tasks)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "by_period" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-07/"
               "NB07_realtime_scheduling.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The controller of the battery container runs four periodic tasks on one processor. Python describes them as objects of the class `Task`, computes the share of the processor that they need and the hyperperiod after which their releases repeat.

**From the formulas to the code.** The utilization $U = \sum_i C_i/T_i$ adds the method `utilization` of every task in the function `utilization`. The hyperperiod $H$, the least common multiple of the periods, is `math.lcm`.

In [ ]:
TASKS = [Task("protection", 1, 5), Task("frequency control", 3, 10), Task("battery management", 5, 20),
         Task("Modbus", 9, 50)]

def utilization(tasks):
    """The share of the processor that the tasks need."""
    return sum(t.utilization() for t in tasks)

print(f"{'task':<20} {'C (ms)':>7} {'T (ms)':>7} {'C/T':>6}")
for t in TASKS:
    print(f"{t.name:<20} {t.C:>7} {t.T:>7} {t.utilization():>6.2f}")
print(f"utilization of the first three tasks: {utilization(TASKS[:3]):.2f}, of all four: {utilization(TASKS):.2f}")
H = math.lcm(*[t.T for t in TASKS])
print(f"hyperperiod: {H} ms, in which the processor is idle for {H * (1 - utilization(TASKS)):.0f} ms")

**Python note.** `TASKS[:3]` is a slice with the first three tasks. `math.lcm(*[...])` unpacks the list of the periods into separate arguments, because `math.lcm` expects each number on its own.

**Exercise A.** A logging task of the controller needs 4 ms every 100 ms. Store the utilization of the four tasks together with it in `u_a`.

In [ ]:
u_a = None   # your answer

check("Exercise A", u_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
u_a = utilization(TASKS + [Task("logging", 4, 100)])

check("Exercise A", u_a)

In [ ]:
# Figure: the utilization of the tasks against the bound of Liu and Layland
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
bottom = 0
for i, t in enumerate(TASKS):
    ax[0].bar(0, t.utilization(), bottom=bottom, color=COLORS[i], label=t.name)
    bottom += t.utilization()
ax[0].axhline(1, color="#1B2430", lw=1)
ax[0].axhline(4 * (2 ** 0.25 - 1), color="#56616D", ls="--", lw=1, label="bound for 4 tasks")
ax[0].set_xlim(-1, 1)
ax[0].set_xticks([])
ax[0].set_ylabel("utilization")
ax[0].legend(fontsize=7)
n = np.arange(1, 11)
ax[1].plot(n, n * (2 ** (1 / n) - 1), "o-", color="#1E6FA8", ms=4)
ax[1].axhline(math.log(2), color="#56616D", ls=":")
ax[1].set_xlabel("number of tasks n")
ax[1].set_ylabel("bound of Liu and Layland")
plt.tight_layout()
plt.show()

**Observe and reflect.** Why is a utilization below 1 necessary but not enough for every deadline to be met? What does the idle time of the hyperperiod tell you about room for another task?

## 2. Rate-monotonic scheduling: Response times &#91;[5](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#ref-5)&#93;

*Lecture: [Rate-monotonic scheduling](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#rate-monotonic)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "utilization" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-07/"
               "NB07_realtime_scheduling.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A rate-monotonic (RM) scheduler gives the highest priority to the task with the shortest period. Python computes the bound of Liu and Layland and the exact response time of each task. It then simulates the scheduler millisecond by millisecond to confirm them.

**From the formulas to the code.** The bound $U \le n(2^{1/n} - 1)$ is `rm_bound`. The response time $R_i = C_i + \sum_{j} \lceil R_i/T_j \rceil C_j$ over the tasks of higher priority is iterated in `response_times` until it stops changing. `simulate` releases a job of every task at each period and runs the ready job with the highest priority for one millisecond, and `report` counts the missed deadlines.

In [ ]:
def rm_bound(n):
    """The utilization bound of Liu and Layland for n tasks under rate-monotonic priorities."""
    return n * (2 ** (1 / n) - 1)

def response_times(tasks):
    """The worst-case response time of each task under RM priorities, by the iteration of Joseph and Pandya."""
    order = sorted(tasks, key=lambda t: t.T)
    result = {}
    for i, task in enumerate(order):
        higher = order[:i]
        R = task.C + sum(h.C for h in higher)
        steps = [R]
        while R <= 10 * task.T:
            R_next = task.C + sum(math.ceil(R / h.T) * h.C for h in higher)
            if R_next == R:
                break
            R = R_next
            steps.append(R)
        finite = R <= 10 * task.T
        result[task.name] = dict(R=R if finite else None, steps=steps, ok=finite and R <= task.T)
    return result

def simulate(tasks, policy="RM", horizon=200, extra=100):
    """A preemptive scheduler in steps of 1 ms: the trace and every job with its release, deadline and finish."""
    jobs = [dict(task=i, release=r, deadline=r + t.T, left=t.C, finish=None)
            for i, t in enumerate(tasks) for r in range(0, horizon + extra, t.T)]
    trace = []
    for now in range(horizon + extra):
        ready = [j for j in jobs if j["release"] <= now and j["left"] > 0]
        if not ready:
            trace.append(None)
            continue
        if policy == "RM":
            job = min(ready, key=lambda j: (tasks[j["task"]].T, j["task"], j["release"]))
        else:                                          # EDF: the earliest absolute deadline first
            job = min(ready, key=lambda j: (j["deadline"], j["task"], j["release"]))
        job["left"] -= 1
        trace.append(job["task"])
        if job["left"] == 0:
            job["finish"] = now + 1
    return trace, jobs

def report(tasks, jobs, horizon=200):
    """The jobs due within the horizon, the missed deadlines and the worst response time of each task."""
    rows = []
    for i, t in enumerate(tasks):
        mine = [j for j in jobs if j["task"] == i and j["deadline"] <= horizon]
        missed = sum(j["finish"] is None or j["finish"] > j["deadline"] for j in mine)
        done = [j["finish"] - j["release"] for j in mine if j["finish"] is not None]
        rows.append(dict(task=t.name, jobs=len(mine), missed=missed,
                         worst_ms=max(done) if len(done) == len(mine) else None))
    return pd.DataFrame(rows)

for n in (3, 4):
    print(f"{n} tasks: utilization {utilization(TASKS[:n]):.2f} against the bound {rm_bound(n):.4f}")
for name, r in response_times(TASKS).items():
    print(f"  {name:<20} steps {r['steps']} -> R = {r['R']} ms")
trace, jobs = simulate(TASKS, "RM")
print(report(TASKS, jobs).to_string(index=False))

**Python note.** `min(ready, key=...)` picks the job with the smallest key. A tuple as the key compares its first elements first, so the period decides, then the number of the task and then the release.

**Exercise B.** The Kalman filter with the offset makes the battery management task need 6 ms. Store its response time under RM in `r_b`.

In [ ]:
r_b = None   # your answer

check("Exercise B", r_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_b = response_times([Task("protection", 1, 5), Task("frequency control", 3, 10),
                       Task("battery management", 6, 20), Task("Modbus", 9, 50)])["battery management"]["R"]

check("Exercise B", r_b)

In [ ]:
# Figure: the rate-monotonic schedule of the four tasks over one hyperperiod
def gantt(ax, tasks, trace, jobs, horizon=100, title=""):
    """Draws the trace as bars, one row per task, with the releases and the missed deadlines."""
    for i, t in enumerate(tasks):
        y = len(tasks) - 1 - i
        ax.broken_barh([(k, 1) for k in range(horizon) if trace[k] == i], (y - 0.35, 0.7), color=COLORS[i])
        for j in jobs:
            if j["task"] == i and j["release"] < horizon:
                ax.plot(j["release"], y + 0.45, marker="v", color="#56616D", ms=4)
                late = j["finish"] is None or j["finish"] > j["deadline"]
                if j["deadline"] <= horizon and late:
                    ax.plot(j["deadline"], y, marker="x", color="#B53F2E", ms=10, mew=2.5)
    ax.set_yticks(range(len(tasks)))
    ax.set_yticklabels([t.name for t in reversed(tasks)], fontsize=8)
    ax.set_xlim(0, horizon)
    ax.set_xlabel("time (ms)")
    ax.set_title(title, fontsize=9)

fig, ax = plt.subplots(figsize=(10, 2.8))
gantt(ax, TASKS, trace, jobs, 100, "rate-monotonic, four tasks, U = 0.93")
plt.tight_layout()
plt.show()

**Observe and reflect.** Where in the schedule is the Modbus task preempted, and by which tasks? Why does the battery management task never wait longer than 10 ms?

## 3. Earliest deadline first: Your runs on the controller bench &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#ref-1)&#93;

*Lecture: [Earliest deadline first](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#edf)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "report" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-07/"
               "NB07_realtime_scheduling.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** An earliest-deadline-first (EDF) scheduler always runs the job whose deadline comes first. Python compares it with RM when the battery management task needs 6 ms and in an overload with 8 ms. It also checks your runs on the controller bench.

**From the formulas to the code.** EDF meets every deadline of periodic tasks with deadlines equal to their periods if and only if $U \le 1$. In `simulate`, `policy="EDF"` orders the ready jobs by their absolute deadline, the release plus the period.

In [ ]:
def task_set(c_bm=5, modbus=True):
    """The tasks of the controller with the chosen execution time of battery management, with or without Modbus."""
    tasks = [Task("protection", 1, 5), Task("frequency control", 3, 10), Task("battery management", c_bm, 20)]
    return tasks + [Task("Modbus", 9, 50)] if modbus else tasks

for c_bm in (6, 8):
    tasks = task_set(c_bm)
    r4 = response_times(tasks)["Modbus"]
    print(f"battery management {c_bm} ms: utilization {utilization(tasks):.2f}, Modbus under RM: "
          + (f"R = {r4['R']} ms after the steps {r4['steps']}" if r4["R"] is not None else "no finite response time"))
    for policy in ("RM", "EDF"):
        rep = report(tasks, simulate(tasks, policy)[1])
        print(f"  {policy:>3}: missed " + ", ".join(f"{r.task} {r.missed} of {r.jobs}" for r in rep.itertuples()))

**Python note.** `return tasks + [...] if modbus else tasks` is a conditional expression: It returns the first value when `modbus` is true and the second one otherwise.

**Exercise C.** Store the number of missed deadlines of the protection task under EDF in the overload with 8 ms for the battery management task in `m_c`.

In [ ]:
m_c = None   # your answer

check("Exercise C", m_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m_c = int(report(task_set(8), simulate(task_set(8), "EDF")[1])["missed"][0])

check("Exercise C", m_c)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lab.html) runs these cases on the controller bench. Paste the copied rows into `MY_RUNS`, and the code runs each of your cases again.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_RUNS = """
"""
if MY_RUNS.strip():
    runs = pd.read_csv(io.StringIO(MY_RUNS.strip()))
    source = "your own runs on the controller bench"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4, 5, 6], "policy": ["RM", "RM", "RM", "EDF", "RM", "EDF"],
                         "c_bm": [5, 5, 6, 6, 8, 8], "modbus": [0, 1, 1, 1, 1, 1],
                         "missed": [0, 0, 2, 0, 4, 24]})
    source = "a simulated student"
def model_missed(r):
    tasks = task_set(int(r.c_bm), bool(r.modbus))
    return int(report(tasks, simulate(tasks, r.policy)[1])["missed"].sum())
runs["utilization"] = [round(utilization(task_set(int(r.c_bm), bool(r.modbus))), 2) for r in runs.itertuples()]
runs["model_missed"] = [model_missed(r) for r in runs.itertuples()]
print("Data:", source)
print(runs[["run", "policy", "c_bm", "modbus", "utilization", "missed", "model_missed"]].to_string(index=False))

In [ ]:
# Figure: RM and EDF with 6 ms for the battery management task
fig, ax = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
tasks = task_set(6)
for axis, policy in zip(ax, ["RM", "EDF"]):
    trace, jobs = simulate(tasks, policy)
    gantt(axis, tasks, trace, jobs, 100, f"{policy}, battery management 6 ms, U = {utilization(tasks):.2f}")
plt.tight_layout()
plt.show()

**Observe and reflect.** Which tasks miss their deadlines in the overload under RM, and which under EDF? Which behaviour would you prefer for the protection logic of the campus, and why?

## 4. The PLC scan cycle: Your runs in the switchboard &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#ref-2)&#93;

*Lecture: [The PLC scan cycle](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#plc-scan)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "task_set" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-07/"
               "NB07_realtime_scheduling.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** When the grid breaker opens, the PLC of the main switchboard must open the car park feeder within 50 ms. Python follows 20 trips through its scan cycle and compares the reactions with the worst case.

**From the formulas to the code.** The scan time $T_{scan} = k + 1$ ms of a program of $k$ thousand instructions is `scan_time`. A trip at the moment $e$ is read by the first scan that starts after $e + t_{in}$ and written at the end of that scan, as computed in `reactions`. The bound $2T_{scan} + t_{in} + t_{out}$ is `worst_case`.

In [ ]:
def scan_time(k):
    """The scan time in ms of a program of k thousand instructions: 1 microsecond each, plus 1 ms for the system."""
    return k + 1.0

def worst_case(k):
    """The longest possible reaction in ms: almost one scan of waiting, one scan and the two delays."""
    return 2 * scan_time(k) + T_IN + T_OUT

def reactions(k, seed, n=20):
    """The reactions in ms to n trips of the grid breaker at random moments, the same as in the interactive lab."""
    T = scan_time(k)
    if T > WATCHDOG_MS:
        return None                                  # the watchdog stops the PLC
    uniform, _ = lab_rng(seed)
    out = []
    for i in range(n):
        e = 1000 * i + 1000 * uniform()              # the input terminal changes at e
        read = math.ceil((e + T_IN) / T) * T          # the first scan that reads the filtered input starts here
        out.append(read + T + T_OUT - e)              # the end of that scan writes the output, then the relay
    return np.array(out)

for k in (12, 19, 20, 25, 30):
    r = reactions(k, 700 + k)
    if r is None:
        print(f"{k:2d} thousand instructions: scan {scan_time(k):.0f} ms > watchdog {WATCHDOG_MS:.0f} ms, PLC stopped")
        continue
    print(f"{k:2d} thousand instructions: scan {scan_time(k):.0f} ms, reactions {r.min():.1f} to {r.max():.1f} ms, "
          f"mean {r.mean():.1f} ms, worst case {worst_case(k):.0f} ms "
          + ("within" if worst_case(k) <= LIMIT_MS else "above") + f" the limit of {LIMIT_MS:.0f} ms")

**Python note.** `math.ceil(x / T) * T` rounds the moment `x` up to the next multiple of the scan time, the start of the next scan. Returning `None` marks a run in which the PLC stopped.

**Exercise D.** Store the largest program, in thousand instructions, whose worst case stays within the limit of 50 ms in `k_d`.

In [ ]:
k_d = None   # your answer

check("Exercise D", k_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
k_d = max(k for k in range(1, 40) if worst_case(k) <= LIMIT_MS)

check("Exercise D", k_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lab.html) opens the PLC cabinet of the main switchboard. Paste the copied rows into `MY_PLC`, and the code repeats each of your runs with the same moments of the trips.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_PLC = """
"""
if MY_PLC.strip():
    plc = pd.read_csv(io.StringIO(MY_PLC.strip()))
    source = "your own runs in the switchboard"
else:
    plc = pd.DataFrame({"run": [1, 2, 3, 4], "program_k": [12, 19, 20, 30], "longest_ms": [34.8, 48.7, 50.6, None]})
    source = "a simulated student"
def model_longest(r):
    out = reactions(int(r.program_k), 700 + int(r.run))
    return None if out is None else round(float(out.max()), 1)
plc["model_longest"] = [model_longest(r) for r in plc.itertuples()]
plc["worst_case"] = [worst_case(int(k)) for k in plc["program_k"]]
print("Data:", source)
print(plc[["run", "program_k", "longest_ms", "model_longest", "worst_case"]].to_string(index=False))

In [ ]:
# Figure: the reactions of the PLC against the size of its program
ks = np.arange(2, 30)
fig, ax = plt.subplots(figsize=(8, 3.4))
for k in ks:
    r = reactions(int(k), 700 + int(k))
    ax.plot([k, k], [r.min(), r.max()], color="#9DB9D3", lw=4)
    ax.plot(k, r.mean(), "o", color="#1E6FA8", ms=3)
ax.plot(ks, [worst_case(int(k)) for k in ks], color="#B53F2E", label="worst case 2 T + 9 ms")
ax.plot(ks, [scan_time(int(k)) + T_IN + T_OUT for k in ks], color="#56616D", ls=":", label="best case T + 9 ms")
ax.axhline(LIMIT_MS, color="#1B2430", ls="--", lw=1, label="limit 50 ms")
ax.set_xlabel("program (thousand instructions)")
ax.set_ylabel("reaction (ms)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why do 20 random trips rarely show the worst case? What would you test to be sure that the requirement of 50 ms holds?

## 5. The campus microgrid, stage 7

*Lecture: [The campus microgrid, stage 7: Real-time tasks of the microgrid controller](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#the-campus-microgrid-stage-7)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "reactions" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-07/"
               "NB07_realtime_scheduling.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 7 checks the timing of the computations of the campus. The controller of the battery container runs four tasks on one processor, and the PLC of the main switchboard sheds the car park after a trip of the grid breaker.

In [ ]:
STAGE7 = {}
STAGE7["utilization of the four tasks"] = round(utilization(TASKS), 2)
STAGE7["bound of Liu and Layland for four tasks"] = round(rm_bound(4), 4)
rt = response_times(TASKS)
STAGE7["response times under RM (ms)"] = [rt[t.name]["R"] for t in TASKS]
STAGE7["battery management 6 ms: response of Modbus under RM (ms)"] = response_times(task_set(6))["Modbus"]["R"]
for c_bm, case in ((6, "battery management 6 ms"), (8, "overload with 8 ms")):
    for policy in ("RM", "EDF"):
        rep = report(task_set(c_bm), simulate(task_set(c_bm), policy)[1])
        STAGE7[f"{case}: missed deadlines under {policy}"] = int(rep["missed"].sum())
STAGE7["overload with 8 ms: missed deadlines of protection under EDF"] = int(
    report(task_set(8), simulate(task_set(8), "EDF")[1])["missed"][0])
STAGE7["PLC: worst reaction with 19 thousand instructions (ms)"] = worst_case(19)
STAGE7["PLC: largest program within 50 ms (thousand instructions)"] = max(
    k for k in range(1, 40) if worst_case(k) <= LIMIT_MS)
STAGE7["PLC: smallest program stopped by the watchdog (thousand instructions)"] = min(
    k for k in range(1, 60) if scan_time(k) > WATCHDOG_MS)
STAGE7["frequency control: response as a share of the delay margin of Week 8 (percent)"] = round(
    100 * rt["frequency control"]["R"] / 1072, 2)
for key, value in STAGE7.items():
    print(f"{key:<72} {value}")

**Python note.** `min(k for k in range(...) if ...)` returns the smallest value of a generator that passes the condition, here the first program size whose scan exceeds the watchdog.

In [ ]:
# Figure: the response times against the deadlines and the worst reaction of the PLC
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
x = np.arange(4)
for shift, c_bm, color in ((-0.2, 5, "#1E6FA8"), (0.2, 6, "#C98510")):
    rt_c = response_times(task_set(c_bm))
    ax[0].bar(x + shift, [rt_c[t.name]["R"] for t in task_set(c_bm)], 0.38, color=color,
              label=f"battery management {c_bm} ms")
ax[0].plot(x, [t.T for t in TASKS], "_", color="#B53F2E", ms=30, mew=3, label="deadline")
ax[0].set_xticks(x)
ax[0].set_xticklabels([t.name for t in TASKS], fontsize=7)
ax[0].set_ylabel("response time under RM (ms)")
ax[0].legend(fontsize=7)
ks = np.arange(2, 36)
ax[1].plot(ks, [worst_case(int(k)) if scan_time(int(k)) <= WATCHDOG_MS else np.nan for k in ks], color="#1E6FA8")
ax[1].axhline(LIMIT_MS, color="#B53F2E", ls="--", label="limit 50 ms")
ax[1].axvline(29.5, color="#56616D", ls=":", label="watchdog stops the PLC")
ax[1].set_xlabel("program (thousand instructions)")
ax[1].set_ylabel("worst reaction (ms)")
ax[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

**Exercise E.** Store the utilization of the four tasks when the battery management task needs 7 ms in `u_e`.

In [ ]:
u_e = None   # your answer

check("Exercise E", u_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
u_e = utilization(task_set(7))

check("Exercise E", u_e)

**Observe and reflect.** The frequency control answers within 4 ms. Why does the delay margin of Week 8 still matter for the loop through the campus server?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE7" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-07/"
               "NB07_realtime_scheduling.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Every embedded system with several periodic duties faces the same questions of utilization and response time. The functions of this week analyse a set of tasks after one change of a name.

The tasks and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "drone"
# choose: drone, car_brakes, robot_arm, printer_3d, wind_turbine, smart_thermostat
APPS = {  # tasks as (name, execution time C in ms, period T in ms)
    "drone": [("attitude control", 1, 4), ("position control", 3, 10), ("radio link", 4, 20), ("camera", 12, 50)],
    "car_brakes": [("wheel speeds", 1, 5), ("brake pressure", 2, 10), ("stability control", 5, 20),
                   ("diagnostics", 10, 100)],
    "robot_arm": [("joint control", 1, 2), ("trajectory", 2, 10), ("vision", 10, 40)],
    "printer_3d": [("stepper pulses", 1, 2), ("temperature control", 5, 50), ("display", 20, 200)],
    "wind_turbine": [("pitch control", 2, 10), ("grid monitoring", 3, 20), ("yaw control", 5, 100),
                     ("logging", 20, 1000)],
    "smart_thermostat": [("sensor reading", 2, 100), ("display", 40, 200), ("radio", 30, 500), ("control", 5, 1000)]}
tasks = [Task(*row) for row in APPS[APP]]
hyper = math.lcm(*[t.T for t in tasks])
print(f"{APP}: utilization {utilization(tasks):.3f}, bound for {len(tasks)} tasks {rm_bound(len(tasks)):.3f}, "
      f"hyperperiod {hyper} ms")
for name, r in response_times(tasks).items():
    T = next(t.T for t in tasks if t.name == name)
    print(f"  {name:<20} R = {r['R']} ms against the deadline of {T} ms -> {'met' if r['ok'] else 'MISSED'}")
for policy in ("RM", "EDF"):
    rep = report(tasks, simulate(tasks, policy, hyper, hyper // 2)[1], hyper)
    print(f"  {policy:>3}: {int(rep['missed'].sum())} missed deadlines in one hyperperiod")

**Python note.** `Task(*row)` unpacks a tuple of three values into the three arguments of the class. The simulation runs one hyperperiod, after which the pattern of releases repeats.

**Observe and reflect.** For which system does RM fail while EDF still meets every deadline? What would you change in that system first: An execution time, a period or the scheduler?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Extend the class `Task` with a deadline shorter than the period and order the tasks by their deadlines. Test the response times against `simulate`. |
| Embedded systems | Add a context switch of 0.1 ms to every preemption in `simulate`, in steps of 0.1 ms. How much utilization does the controller lose? |
| Internet of Things | A sensor node wakes up every 500 ms, works for 8 ms and sleeps otherwise. Estimate the share of time awake and the life of a battery of 2,000 mAh at 20 mA awake and 0.01 mA asleep. |
| Computer networks | The Modbus task waits for a reply of 30 ms without using the processor. Split it into two jobs before and after the reply and repeat the analysis. |
| Control engineering | Delay the output of the frequency control of Week 5 by its response time of 4 ms and by 40 ms. How do the nadir and the settling time change? |
| Data science | Draw 1,000 random execution times of the battery management task between 4 and 6 ms and estimate the probability that the Modbus task misses its deadline under RM. |
| Cybersecurity | An attacker floods the controller with Modbus requests that double the execution time of the Modbus task. Which tasks miss their deadlines under RM and under EDF? |
| Artificial intelligence | Count the multiply-accumulate operations of a network with 10 inputs, two hidden layers of 32 neurons and one output. Estimate its execution time at 100 million operations per second and add it as a task. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-07).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>